In [2]:
# %%
# ============================================================
# ☀️ Full-Disk AIA + HMI Downloader  → .NPZ Converter (Mac Optimized)
# ============================================================
# • Downloads full-disk AIA + all HMI products for 10 flares
# • Uses 6 timestamps (−24, −18, −12, −6, 0, +6 h)
# • Automatically picks nearest FITS (±15 min)
# • Converts to 256×256 .npz and deletes FITS immediately
# ============================================================

import os, glob, cv2
import numpy as np
from datetime import datetime, timedelta
from dotenv import load_dotenv
from astropy.io import fits
import astropy.units as u
from sunpy.net import Fido, attrs as a

# ============================================================
# 🔧 Setup
# ============================================================

load_dotenv()
EMAIL = os.environ.get("EMAIL")
if not EMAIL:
    raise ValueError("❌ Missing EMAIL in .env – Add EMAIL=your_registered_email@njit.edu")

BASE_DIR = os.path.abspath(".")
print(f"✅ Base directory: {BASE_DIR}")
print(f"✅ Using JSOC email: {EMAIL}")

✅ Base directory: /Users/sathviksoman/Downloads/Surya/downstream_examples/euv_spectra_prediction/assets/flare_data
✅ Using JSOC email: ss5369@njit.edu


/opt/miniconda3/envs/surya_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 🔆 10 Selected Flares (Start times)
# ============================================================

FLARES = [
    ("AR11158_X2.2","2011-02-15 01:44:00"),
    ("AR11429_X5.4","2012-03-07 00:02:00"),
    ("AR11520_X1.4","2012-07-12 15:37:00"),
    ("AR12158_X1.6","2014-09-10 17:21:00"),
    ("AR12192_X3.1","2014-10-24 21:07:00"),
    ("AR11158_M6.6","2011-02-13 17:28:00"),
    ("AR11261_M9.3","2011-07-30 02:04:00"),
    ("AR11429_M6.3","2012-03-09 03:22:00"),
    ("AR11719_M6.5","2013-04-11 06:55:00"),
    ("AR12036_M7.3","2014-04-18 12:31:00"),
]


In [5]:
# ============================================================
# 💾 FITS → NPZ converter + cleanup
# ============================================================

def fits_to_npz_and_cleanup(save_dir, out_npz, target_size=(256, 256)):
    """Combine all FITS in folder → .NPZ and delete FITS."""
    stacks = {}
    fits_files = glob.glob(os.path.join(save_dir,"*.fits"))
    if not fits_files:
        print(f"⚠️ No FITS in {save_dir}")
        return

    for fpath in fits_files:
        try:
            data = fits.getdata(fpath).astype(np.float32)
            lo, hi = np.percentile(data, 1), np.percentile(data, 99)
            data = np.clip(data, lo, hi)
            data = np.log1p(data - lo + 1e-6)
            data = cv2.resize(data, target_size)

            # Infer channel name
            ch = "unknown"
            if "aia" in fpath:
                for wl in ["94","131","171","193","211","304","335","1600"]:
                    if f".{wl}" in fpath or f"_{wl}." in fpath:
                        ch = f"aia{wl}"
                        break
            elif "hmi.B_720s" in fpath:
                if ".field" in fpath: ch = "hmiB_field"
                elif ".inclination" in fpath: ch = "hmiB_incl"
                elif ".azimuth" in fpath: ch = "hmiB_azim"
            elif "hmi.M_720s" in fpath: ch = "hmiM"
            elif "hmi.ic_720s" in fpath: ch = "hmiIC"
            stacks.setdefault(ch, []).append(data)
        except Exception as e:
            print("⚠️ Error reading", fpath, e)

    packed = {k: np.stack(v, axis=0) for k,v in stacks.items() if len(v)>0}
    np.savez_compressed(out_npz, **packed)
    print(f"💾 Saved {out_npz} ({len(packed)} channels)")
    for fpath in fits_files:
        try: os.remove(fpath)
        except: pass
    print(f"🧹 Deleted {len(fits_files)} FITS from {save_dir}")

In [6]:
# ============================================================
# 📥 JSOC Download function (AIA + full HMI)
# ============================================================

def download_from_jsoc(start_time, end_time, save_dir, email):
    os.makedirs(save_dir, exist_ok=True)

    # AIA EUV + UV
    aia_series = [
        ("aia.lev1_euv_12s","image",[94,131,171,193,211,304,335]),
        ("aia.lev1_uv_24s","image",[1600]),
    ]
    for series, seg, waves in aia_series:
        for w in waves:
            print(f"🔹 AIA {w}Å: {start_time} → {end_time}")
            try:
                query = Fido.search(
                    a.Time(start_time, end_time),
                    a.jsoc.Series(series),
                    a.Sample(12*u.minute),
                    a.jsoc.PrimeKey('WAVELNTH', w),
                    a.jsoc.Segment(seg),
                    a.jsoc.Notify(email)
                )
                if len(query)>0: Fido.fetch(query, path=save_dir)
                else: print(f"⚠️ No data for AIA {w}Å")
            except Exception as e: print(f"⚠️ AIA {w}Å failed: {e}")

    # HMI (all channels)
    hmi_series = [
        ("hmi.B_720s","field"),
        ("hmi.B_720s","inclination"),
        ("hmi.B_720s","azimuth"),
        ("hmi.M_720s",None),
        ("hmi.ic_720s",None),
    ]
    for series, seg in hmi_series:
        print(f"🔹 HMI {series} {seg or ''}: {start_time} → {end_time}")
        try:
            attrs = [
                a.Time(start_time, end_time),
                a.jsoc.Series(series),
                a.Sample(12*u.minute),
                a.jsoc.Notify(email)
            ]
            if seg: attrs.append(a.jsoc.Segment(seg))
            query = Fido.search(*attrs)
            if len(query)>0: Fido.fetch(query, path=save_dir)
            else: print(f"⚠️ No data for {series} {seg or ''}")
        except Exception as e: print(f"⚠️ HMI {series} failed: {e}")

In [7]:
# ============================================================
# 🧭 Generate time ranges and download + convert
# ============================================================

for flare_name, flare_start in FLARES:
    flare_dir = os.path.join(BASE_DIR, flare_name, "full_disk")
    os.makedirs(flare_dir, exist_ok=True)

    flare_start_dt = datetime.strptime(flare_start, "%Y-%m-%d %H:%M:%S")
    offsets = [-24, -18, -12, -6, 0, 6]  # ✅ 6 timestamps

    print(f"\n===============================================")
    print(f"☀️ Processing {flare_name} ({flare_start})")
    print("===============================================")

    for h in offsets:
        t0 = flare_start_dt + timedelta(hours=h)
        t1 = t0 + timedelta(minutes=30)
        save_dir = os.path.join(flare_dir, f"{t0.strftime('%Y%m%dT%H%M')}")
        os.makedirs(save_dir, exist_ok=True)

        print(f"\n🕓 Download window: {t0} → {t1}")
        download_from_jsoc(
            start_time=t0.strftime("%Y-%m-%dT%H:%M:%S"),
            end_time=t1.strftime("%Y-%m-%dT%H:%M:%S"),
            save_dir=save_dir,
            email=EMAIL
        )

        # 💾 Convert → .NPZ + delete FITS
        out_npz = os.path.join(flare_dir, f"{t0.strftime('%Y%m%dT%H%M')}.npz")
        fits_to_npz_and_cleanup(save_dir, out_npz, target_size=(256, 256))

print("\n🎯 All 10 flares processed (6 timestamps each) → .NPZ files (256×256).")


☀️ Processing AR11158_X2.2 (2011-02-15 01:44:00)

🕓 Download window: 2011-02-14 01:44:00 → 2011-02-14 02:14:00
🔹 AIA 94Å: 2011-02-14T01:44:00 → 2011-02-14T02:14:00


2025-10-31 16:33:00 - drms - INFO: Export request pending. [id=JSOC_20251031_003067, status=2]
2025-10-31 16:33:00 - drms - INFO: Waiting for 0 seconds...
2025-10-31 16:33:00 - sunpy - INFO: 3 URLs found for download. Full request totaling 21MB


INFO: 3 URLs found for download. Full request totaling 21MB [sunpy.net.jsoc.jsoc]


Files Downloaded: 100%|██████████| 3/3 [00:11<00:00,  3.72s/file]


🔹 AIA 131Å: 2011-02-14T01:44:00 → 2011-02-14T02:14:00


...

🔹 HMI hmi.ic_720s : 2014-04-18T18:31:00 → 2014-04-18T19:01:00


2025-11-01 03:05:01 - drms - INFO: Export request pending. [id=JSOC_20251101_001260, status=2]
2025-11-01 03:05:01 - drms - INFO: Waiting for 0 seconds...
2025-11-01 03:05:01 - drms - INFO: Export request pending. [id=JSOC_20251101_001260, status=1]
2025-11-01 03:05:01 - drms - INFO: Waiting for 5 seconds...
2025-11-01 03:05:06 - drms - INFO: Export request pending. [id=JSOC_20251101_001260, status=1]
2025-11-01 03:05:06 - drms - INFO: Waiting for 5 seconds...
2025-11-01 03:05:12 - drms - INFO: Export request pending. [id=JSOC_20251101_001260, status=1]
2025-11-01 03:05:12 - drms - INFO: Waiting for 5 seconds...
2025-11-01 03:05:17 - drms - INFO: Export request pending. [id=JSOC_20251101_001260, status=1]
2025-11-01 03:05:17 - drms - INFO: Waiting for 5 seconds...
2025-11-01 03:05:23 - sunpy - INFO: 3 URLs found for download. Full request totaling 43MB


INFO: 3 URLs found for download. Full request totaling 43MB [sunpy.net.jsoc.jsoc]


Files Downloaded: 100%|██████████| 3/3 [01:34<00:00, 31.36s/file]


💾 Saved /Users/sathviksoman/Downloads/Surya/downstream_examples/euv_spectra_prediction/assets/flare_data/AR12036_M7.3/full_disk/20140418T1831.npz (10 channels)
🧹 Deleted 39 FITS from /Users/sathviksoman/Downloads/Surya/downstream_examples/euv_spectra_prediction/assets/flare_data/AR12036_M7.3/full_disk/20140418T1831

🎯 All 10 flares processed (6 timestamps each) → .NPZ files (256×256).


In [4]:
import os
print(os.getcwd())


/Users/sathviksoman/Downloads/Surya/downstream_examples/euv_spectra_prediction/assets/flare_data


In [5]:
import numpy as np
f = np.load("AR11158_X2.2/full_disk/20110214T0144.npz")
print(f.keys())
print({k: v.shape for k,v in f.items()})


KeysView(NpzFile 'AR11158_X2.2/full_disk/20110214T0144.npz' with keys: unknown, aia131, aia94, hmiIC, aia335...)
{'unknown': (12, 256, 256), 'aia131': (3, 256, 256), 'aia94': (3, 256, 256), 'hmiIC': (3, 256, 256), 'aia335': (3, 256, 256), 'aia304': (3, 256, 256), 'aia193': (3, 256, 256), 'aia1600': (3, 256, 256), 'aia211': (3, 256, 256), 'aia171': (3, 256, 256)}


In [1]:
import os, glob, numpy as np
from prettytable import PrettyTable

table = PrettyTable(["Flare", "Timestamp", "Num Channels", "Channels"])

# --- Loop through each flare directory ---
for flare_dir in sorted(glob.glob("AR*")):
    npz_files = sorted(glob.glob(os.path.join(flare_dir, "full_disk", "*.npz")))
    if not npz_files:
        continue  # skip empty dirs
    
    for npz_path in npz_files:
        try:
            f = np.load(npz_path)
            keys = list(f.keys())
            table.add_row([
                os.path.basename(flare_dir),
                os.path.basename(npz_path).replace(".npz",""),
                len(keys),
                ", ".join(keys)
            ])
            f.close()
        except Exception as e:
            table.add_row([
                os.path.basename(flare_dir),
                os.path.basename(npz_path),
                "Error",
                str(e)
            ])

# --- Display summary ---
print(table)


+--------------+---------------+--------------+--------------------------------------------------------------------------------+
|    Flare     |   Timestamp   | Num Channels |                                    Channels                                    |
+--------------+---------------+--------------+--------------------------------------------------------------------------------+
| AR11158_M6.6 | 20110212T1728 |      10      | aia335, hmiIC, aia131, unknown, aia304, aia1600, aia193, aia211, aia171, aia94 |
| AR11158_M6.6 | 20110212T2328 |      10      | aia94, aia1600, unknown, aia193, aia171, hmiIC, aia211, aia335, aia131, aia304 |
| AR11158_M6.6 | 20110213T0528 |      10      | hmiIC, aia94, aia171, aia211, aia304, unknown, aia335, aia193, aia1600, aia131 |
| AR11158_M6.6 | 20110213T1128 |      10      | aia211, hmiIC, unknown, aia171, aia1600, aia94, aia193, aia304, aia131, aia335 |
| AR11158_M6.6 | 20110213T1728 |      10      | aia335, unknown, aia1600, aia131, aia211, hmiIC, 

In [2]:
import os, glob, numpy as np

print(f"{'Flare':<20} {'Timestamp':<15} {'Has_hmiM':<10} Channels")
print("="*90)

total_files = 0
has_hmiM_count = 0

for flare_dir in sorted(glob.glob("AR*")):
    npz_files = sorted(glob.glob(os.path.join(flare_dir, "full_disk", "*.npz")))
    if not npz_files:
        continue
    
    for npz_path in npz_files:
        try:
            f = np.load(npz_path)
            keys = list(f.keys())
            has_hmiM = "hmiM" in keys
            total_files += 1
            if has_hmiM:
                has_hmiM_count += 1
            print(f"{os.path.basename(flare_dir):<20} {os.path.basename(npz_path).replace('.npz',''):<15} {str(has_hmiM):<10} {', '.join(keys)}")
            f.close()
        except Exception as e:
            print(f"{os.path.basename(flare_dir):<20} ERROR {e}")

print("="*90)
print(f"✅ Summary: {has_hmiM_count}/{total_files} NPZ files contain 'hmiM' channel.")


Flare                Timestamp       Has_hmiM   Channels
AR11158_M6.6         20110212T1728   False      aia335, hmiIC, aia131, unknown, aia304, aia1600, aia193, aia211, aia171, aia94
AR11158_M6.6         20110212T2328   False      aia94, aia1600, unknown, aia193, aia171, hmiIC, aia211, aia335, aia131, aia304
AR11158_M6.6         20110213T0528   False      hmiIC, aia94, aia171, aia211, aia304, unknown, aia335, aia193, aia1600, aia131
AR11158_M6.6         20110213T1128   False      aia211, hmiIC, unknown, aia171, aia1600, aia94, aia193, aia304, aia131, aia335
AR11158_M6.6         20110213T1728   False      aia335, unknown, aia1600, aia131, aia211, hmiIC, aia304, aia171, aia193, aia94
AR11158_M6.6         20110213T2328   False      aia131, unknown, aia193, aia335, aia304, aia211, aia94, aia1600, aia171, hmiIC
AR11158_X2.2         20110214T0144   False      unknown, aia131, aia94, hmiIC, aia335, aia304, aia193, aia1600, aia211, aia171
AR11158_X2.2         20110214T0744   False      aia304

In [3]:
import os, glob, numpy as np

count_fixed = 0
total = 0

for flare_dir in sorted(glob.glob("AR*")):
    npz_files = sorted(glob.glob(os.path.join(flare_dir, "full_disk", "*.npz")))
    for npz_path in npz_files:
        total += 1
        data = np.load(npz_path)
        keys = list(data.keys())
        if "unknown" in keys and "hmiM" not in keys:
            # Copy data
            arrays = {k: data[k] for k in keys}
            arrays["hmiM"] = arrays.pop("unknown")  # rename key
            # Save back
            np.savez_compressed(npz_path, **arrays)
            count_fixed += 1
            print(f"✅ Renamed 'unknown' → 'hmiM' in {npz_path}")
        data.close()

print(f"\n🎯 Fixed {count_fixed} files out of {total} total.")


✅ Renamed 'unknown' → 'hmiM' in AR11158_M6.6/full_disk/20110212T1728.npz
✅ Renamed 'unknown' → 'hmiM' in AR11158_M6.6/full_disk/20110212T2328.npz
✅ Renamed 'unknown' → 'hmiM' in AR11158_M6.6/full_disk/20110213T0528.npz
✅ Renamed 'unknown' → 'hmiM' in AR11158_M6.6/full_disk/20110213T1128.npz
✅ Renamed 'unknown' → 'hmiM' in AR11158_M6.6/full_disk/20110213T1728.npz
✅ Renamed 'unknown' → 'hmiM' in AR11158_M6.6/full_disk/20110213T2328.npz
✅ Renamed 'unknown' → 'hmiM' in AR11158_X2.2/full_disk/20110214T0144.npz
✅ Renamed 'unknown' → 'hmiM' in AR11158_X2.2/full_disk/20110214T0744.npz
✅ Renamed 'unknown' → 'hmiM' in AR11158_X2.2/full_disk/20110214T1344.npz
✅ Renamed 'unknown' → 'hmiM' in AR11158_X2.2/full_disk/20110214T1944.npz
✅ Renamed 'unknown' → 'hmiM' in AR11158_X2.2/full_disk/20110215T0144.npz
✅ Renamed 'unknown' → 'hmiM' in AR11158_X2.2/full_disk/20110215T0744.npz
✅ Renamed 'unknown' → 'hmiM' in AR11261_M9.3/full_disk/20110729T0204.npz
✅ Renamed 'unknown' → 'hmiM' in AR11261_M9.3/full_d

In [4]:
import os, glob, numpy as np

print(f"{'Flare':<20} {'Timestamp':<15} {'Has_hmiM':<10} Channels")
print("="*90)

for flare_dir in sorted(glob.glob("AR*")):
    npz_files = sorted(glob.glob(os.path.join(flare_dir, "full_disk", "*.npz")))
    for npz_path in npz_files:
        f = np.load(npz_path)
        keys = list(f.keys())
        print(f"{os.path.basename(flare_dir):<20} {os.path.basename(npz_path).replace('.npz',''):<15} {str('hmiM' in keys):<10} {', '.join(keys)}")
        f.close()


Flare                Timestamp       Has_hmiM   Channels
AR11158_M6.6         20110212T1728   True       aia335, hmiIC, aia131, aia304, aia1600, aia193, aia211, aia171, aia94, hmiM
AR11158_M6.6         20110212T2328   True       aia94, aia1600, aia193, aia171, hmiIC, aia211, aia335, aia131, aia304, hmiM
AR11158_M6.6         20110213T0528   True       hmiIC, aia94, aia171, aia211, aia304, aia335, aia193, aia1600, aia131, hmiM
AR11158_M6.6         20110213T1128   True       aia211, hmiIC, aia171, aia1600, aia94, aia193, aia304, aia131, aia335, hmiM
AR11158_M6.6         20110213T1728   True       aia335, aia1600, aia131, aia211, hmiIC, aia304, aia171, aia193, aia94, hmiM
AR11158_M6.6         20110213T2328   True       aia131, aia193, aia335, aia304, aia211, aia94, aia1600, aia171, hmiIC, hmiM
AR11158_X2.2         20110214T0144   True       aia131, aia94, hmiIC, aia335, aia304, aia193, aia1600, aia211, aia171, hmiM
AR11158_X2.2         20110214T0744   True       aia304, aia1600, hmiIC, aia

In [1]:
from astropy.io import fits
hdu = fits.open("azimuth.fits")
print(hdu[0].data.shape)
print(hdu[0].header['CONTENT'])
print(hdu[0].header['BUNIT'])


AttributeError: 'NoneType' object has no attribute 'shape'

In [2]:
from astropy.io import fits

hdr = fits.getheader("azimuth.fits")
print("NAXIS:", hdr.get("NAXIS"))
print("CONTENT:", hdr.get("CONTENT"))
print("BITPIX:", hdr.get("BITPIX"))
print("COMMENT:", hdr.get("COMMENT")[:5])


NAXIS: 0
CONTENT: None
BITPIX: 16
COMMENT:   FITS (Flexible Image Transport System) format is defined in 'Astronomy
  and Astrophysics', volume 376, page 359; bibcode: 2001A&A...376..359H
